# Feature Engineering

В репозиторий не попала первая версия baseline-модели.

В первой версии baseline были созданы дополнительные признаки пользователей и постов:

### User features:
- total_likes
- views
- share_likes_views (отношение лайков к просмотрам)
- fav_category_likes (любимая категория по лайкам)
- ~~fav_category_views (любимая категория по просмотрам)~~
- ~~share_category_likes (доля лайков по категориям)~~
- ~~share_category_views (доля просмотров по категориям)~~
- share_time_likes (доля лайков по времени суток)
- ~~share_time_views (доля просмотров по времени суток)~~

Encoding категориальных признаков:
- OHE:
  - country
  - exp_group
  - os
  - source
  - fav_category_likes
  - ~~fav_category_views~~

- MHE:
  - city_clean

### Post features:
- topic_post (OHE)
- count_likes_post
- count_views_post
- share_likes_views_post
- TF-IDF признаки текста
- long_text
- count_word
- unique_count_word
- ~~доля лайков по части дня~~
- ~~доля просмотров по части дня~~

После формирования baseline была обучена первая модель — Logistic Regression.

Далее был проведён анализ весов признаков модели. Признаки, отмеченные выше, имели низкий вклад в предсказание (малые значения весов модели) и были исключены из дальнейшего обучения.

В текущем ноутбуке используется обновлённый набор признаков без малоинформативных параметров. В нём выполняется повторное формирование признаков, заполнение пропусков и подготовка итогового датасета для дальнейшего обучения и сравнения моделей.


In [10]:
import pandas as pd

In [11]:
user_data = pd.read_csv("../data/raw/user_data.csv")
feed_data = pd.read_csv("../data/raw/feed_data.csv")
post_text_df = pd.read_csv("../data/raw/post_text_df.csv")

### Сортируем данные по времени, чтобы разделение на train и test учитывало временную последовательность событий.
Это позволяет избежать утечки данных из будущего в прошлое.

In [12]:
feed_data = feed_data.sort_values("timestamp")
n_total = len(feed_data)
n_test = int(n_total * 0.3)  # 30% для теста
n_train = n_total - n_test
train = feed_data.iloc[:n_train].reset_index(drop=True)
test  = feed_data.iloc[n_train:].reset_index(drop=True)

### Добавление пользовательских признаков

Для каждого пользователя рассчитываем количество лайков и просмотров на основе истории взаимодействий.

- `total_likes` — количество постов, которые пользователь лайкнул.
- `views` — количество просмотренных постов без лайка.

In [13]:
total_likes = train.groupby('user_id')['target'].sum().reset_index(name="total_likes")
views_only = (
    (1 - train["target"])
    .groupby(train["user_id"])
    .sum()
    .reset_index(name="views")
)
user_data = (user_data.merge(views_only, on='user_id', how='left'))
user_data = (user_data.merge(total_likes, on='user_id', how='left'))

### Соотношение лайков и просмотров пользователя

Создаём признак `share_likes_views`, который показывает долю лайков среди всех взаимодействий пользователя.

In [14]:
user_data["share_likes_views"] = user_data["total_likes"]/user_data["views"]

### Любимая категория пользователя

Создаём признак `fav_category_likes` — категорию постов, которые пользователь чаще всего лайкал.

In [15]:
train = train.merge(post_text_df[["post_id", "topic"]], on="post_id", how="left")
agg_likes = train.pivot_table(index='user_id', columns='topic', 
                      values='action', 
                      aggfunc=lambda x: (x=='like').sum())
train = train.drop(columns='topic')
fav_category_likes = agg_likes.idxmax(axis=1).reset_index().rename(columns ={0:"fav_category_likes"})
user_data = user_data.merge(fav_category_likes, on = "user_id", how="left")

### Доля лайков пользователя по времени суток

Создаём признаки, которые показывают, в какое время суток пользователь чаще взаимодействует с контентом.

Сначала разделяем время взаимодействия на четыре периода:
- morning — утро (6:00–12:00)
- afternoon — день (12:00–18:00)
- evening — вечер (18:00–24:00)
- night — ночь (00:00–6:00)

После этого рассчитываем долю лайков пользователя в каждом временном периоде относительно всех его лайков.

In [16]:
train['timestamp'] = pd.to_datetime(train['timestamp'])
train['hour'] = train['timestamp'].dt.hour
train['week'] = train['timestamp'].dt.dayofweek
train['month'] = train['timestamp'].dt.month
def time_of_day(hour):
    if 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 24:
        return 'evening'
    else:
        return 'night'

train['part_of_day'] = train['hour'].apply(time_of_day)
# Считаем количество просмотров и лайков для каждого пользователя по части дня
agg_likes_day = train.pivot_table(index='user_id', columns='part_of_day', 
                      values='action', 
                      aggfunc=lambda x: (x=='like').sum())
share_time_likes = agg_likes_day.div(agg_likes_day.sum(axis=1), axis=0).add_prefix('share_likes_time_')
user_data = user_data.merge(share_time_likes, on = "user_id", how="left")

### Mean Target Encoding признака города

Признак `city` преобразуется в числовой формат с помощью Mean Target Encoding.

Для каждого города рассчитывается средняя вероятность взаимодействия пользователя с контентом (`target`). Чтобы избежать переобучения на городах с небольшим количеством данных, применяется сглаживание с использованием общего среднего значения по датасету.

После расчёта:
- полученный признак `city_encoded` добавляется в таблицу пользователей;
- исходный категориальный признак `city` удаляется;
- пропущенные значения заменяются средним значением признака.

В результате вместо категориального признака города модель получает числовой признак, отражающий поведение пользователей из данного города.

In [17]:
city_stats = train.merge(user_data[["user_id", "city"]], on="user_id", how="left").groupby('city')['target'].agg(['mean', 'count'])
city_stats.columns = ['city_mean', 'city_count']
m = 10  # степень сглаживания (можно 5–100)

global_mean = train['target'].mean()

city_stats['city_encoded'] = (
    city_stats['city_count'] * city_stats['city_mean'] +
    m * global_mean
) / (city_stats['city_count'] + m)

city_stats = city_stats.reset_index()
user_data = user_data.merge(city_stats[['city', 'city_encoded']], on='city', how='left')
user_data = user_data.drop(columns=["city"])
user_data['city_encoded'] = user_data['city_encoded'].fillna(user_data['city_encoded'].mean())

### Заполнение пропущенных значений в пользовательских признаках

После генерации признаков в данных остаются пропущенные значения.

Для числовых признаков выполняется групповое заполнение:
- пользователи объединяются по основным характеристикам (`gender`, `age`, `country`, `exp_group`, `os`, `source`, `city_encoded`);
- пропуски заменяются средним значением признака внутри похожей группы пользователей.

Для категориального признака `fav_category_likes` пропуски заполняются наиболее часто встречающейся категорией внутри группы.

Если после группового заполнения остаются пропуски, используется глобальное заполнение:
- числовые признаки → среднее значение по всему датасету;
- категориальные признаки → наиболее частая категория.

In [18]:
# Признаки, по которым определяем похожих пользователей
group_columns = ['gender', 'age', 'country', 'exp_group', 'os', 'source','city_encoded']

# Числовые признаки для заполнения пропусков
num_col = ['views', 'total_likes', 'share_likes_views'
           , 'share_likes_time_afternoon',
       'share_likes_time_evening', 'share_likes_time_morning']

# Заполняем пропуски средним значением внутри групп похожих пользователей
user_data[num_col] = user_data.groupby(group_columns)[num_col].transform(
    lambda x: x.fillna(x.mean())
)

# Заполняем пропуски любимой категории модой внутри группы
user_data['fav_category_likes'] = user_data.groupby(group_columns)['fav_category_likes'].transform(
    lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x
)

# Если после группового заполнения остались пропуски — используем глобальную моду
user_data['fav_category_likes'] = user_data['fav_category_likes'].fillna(
    user_data['fav_category_likes'].mode()[0]
)

# Финальное заполнение оставшихся пропусков числовых признаков средним значением
user_data[num_col] = user_data[num_col].fillna(user_data[num_col].mean())

### One-Hot Encoding категориальных признаков пользователей

Для преобразования категориальных признаков в числовой формат используется One-Hot Encoding.

Бинарные признаки создаются для следующих категорий:
- country
- exp_group
- os
- source
- fav_category_likes

In [19]:
OHE_USER = ['country','exp_group','os','source','fav_category_likes'] 
user_data = pd.get_dummies(user_data, 
                           columns=OHE_USER, 
                           drop_first=True)  

### Добавление признаков постов

### Количество лайков у поста

Создаём признак популярности поста — количество полученных лайков в обучающей выборке.

Признак `count_likes_post` показывает, сколько раз пользователи взаимодействовали с постом положительно (`target = 1`).

In [22]:
tota_like_post = train.groupby(["post_id"])["target"].sum().reset_index()
post_text_df = (post_text_df.merge(tota_like_post, on='post_id', how='left')).rename(columns = {"target":"count_likes_post"})

### Количество просмотров поста

Создаём признак количества просмотров каждого поста.

Так как в данных `target = 1` соответствует лайку, а `target = 0` — просмотру без лайка, суммируем значения `(1 - target)` для каждого `post_id`.

In [23]:
views_only = (
    (1 - train["target"])
    .groupby(train["post_id"])
    .sum()
    .reset_index(name="count_views_post")
)
post_text_df = (post_text_df.merge(views_only, on='post_id', how='left'))

### Доля лайков среди просмотров поста

Признак `share_likes_views_post` показывает, какая доля пользователей после просмотра поста поставила лайк.

In [24]:
post_text_df["share_likes_views_post"] = post_text_df["count_likes_post"] / post_text_df["count_views_post"]

### Заполнение пропусков в признаках постов

После объединения статистических признаков с таблицей постов часть значений может отсутствовать для некоторых объектов.

Для заполнения пропусков используются средние значения признаков внутри каждой тематики поста (`topic`), так как посты одной категории могут иметь похожие характеристики.

Заполняются следующие признаки:
- `count_likes_post` — количество лайков поста
- `count_views_post` — количество просмотров поста
- `share_likes_views_post` — отношение лайков к просмотрам

In [25]:
cols = ['count_likes_post', 'count_views_post', 'share_likes_views_post']

for col in cols:
    post_text_df[col] = post_text_df[col].fillna(
        post_text_df.groupby('topic')[col].transform('mean')
    )

### TF-IDF признаки текста постов

Для получения признаков из текста используется метод TF-IDF.

Сначала обучается `TfidfVectorizer` на текстах постов, после чего каждый текст преобразуется в разреженный вектор признаков.

Так как полученные TF-IDF векторы имеют большую размерность, вместо использования всех отдельных слов создаются агрегированные признаки:

- `tfidf_mean` — среднее значение TF-IDF весов по всем словам поста;
- `tfidf_max` — максимальное значение TF-IDF веса среди слов поста;
- `tfidf_std` — стандартное отклонение TF-IDF весов, характеризующее распределение значимости слов в тексте.

Эти признаки позволяют учитывать текстовую информацию постов без значительного увеличения размерности модели.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Зафиттим наши данные в TfidfVectorizer
tfidf = TfidfVectorizer()
tfidf.fit(post_text_df['text'])
X_tfidf = tfidf.transform(post_text_df['text'])

tfidf_mean = X_tfidf.mean(axis=1)
tfidf_max = X_tfidf.max(axis=1).toarray()
post_text_df["tfidf_mean"] = np.array(tfidf_mean).ravel()
post_text_df["tfidf_max"] = np.array(tfidf_max).ravel()
mean = X_tfidf.mean(axis=1)
mean_sq = X_tfidf.multiply(X_tfidf).mean(axis=1)

tfidf_std = np.sqrt(mean_sq - np.square(mean))
post_text_df["tfidf_std"] = np.array(tfidf_std).ravel()

### Статистические признаки текста постов

Создаются дополнительные признаки, описывающие структуру текста поста:

- `long_text` — длина текста в символах;
- `count_word` — количество слов в тексте;
- `unique_count_word` — количество уникальных слов в тексте.

После формирования признаков исходный текст удаляется, так как он больше не используется напрямую в модели.

In [29]:
post_text_df["long_text"] = post_text_df["text"].apply(len)
post_text_df["coun_word"] = post_text_df["text"].str.split().str.len()
post_text_df["unique_count_word"] = post_text_df["text"].str.split().apply(lambda x: len(set(x)))
post_text_df = post_text_df.drop(columns='text')

### Кодирование категории поста

Категориальный признак `topic` преобразуется в числовой формат с помощью One-Hot Encoding.

In [31]:
post_text_df = pd.get_dummies(post_text_df, columns=['topic'], drop_first=True)  

### Формирование итоговых датасетов для обучения модели

На данном этапе объединяются все подготовленные признаки пользователей, постов и временные признаки.

В качестве временных признаков используются циклические преобразования:
- `hour_sin`, `hour_cos` — время суток;
- `week_sin`, `week_cos` — день| недели;
- `month_sin`, `month_cos` — месяц.


После формирования временных признаков:
1. Объединяются признаки взаимодействий пользователя и поста.
2. Добавляются пользовательские признаки (`user_data`).
3. Добавляются признаки постов (`post_text_df`).
4. Удаляются идентификаторы `user_id` и `post_id`, так как они не используются как признаки модели.
5. Числовые признаки переводятся в формат `float32` для уменьшения потребления памяти.

In [36]:
train['hour_sin'] = np.sin(2 * np.pi * train['hour'] / 24)
train['hour_cos'] = np.cos(2 * np.pi * train['hour'] / 24)
train['month_sin'] = np.sin(2 * np.pi * train['month'] / 12)
train['month_cos'] = np.cos(2 * np.pi * train['month'] / 12)
train['week_sin'] = np.sin(2 * np.pi * train['week'] / 7)
train['week_cos'] = np.cos(2 * np.pi * train['week'] / 7)

result = train[['hour_sin','hour_cos','month_sin','month_cos', 'week_sin', 'week_cos','user_id', 'post_id', 'target']]

done_train = result.merge(user_data, on='user_id', how='left').merge(post_text_df, on='post_id', how='left')
done_train = done_train.drop(columns=['user_id','post_id'])

test['timestamp'] = pd.to_datetime(test['timestamp'])
test['hour'] = test['timestamp'].dt.hour
test['week'] = test['timestamp'].dt.dayofweek
test['month'] = test['timestamp'].dt.month
test['hour_sin'] = np.sin(2 * np.pi * test['hour'] / 24)
test['hour_cos'] = np.cos(2 * np.pi * test['hour'] / 24)
test['month_sin'] = np.sin(2 * np.pi * test['month'] / 12)
test['month_cos'] = np.cos(2 * np.pi * test['month'] / 12)
test['week_sin'] = np.sin(2 * np.pi * test['week'] / 7)
test['week_cos'] = np.cos(2 * np.pi * test['week'] / 7)

result_test = test[[ 'hour_sin','hour_cos','month_sin','month_cos', 'week_sin', 'week_cos','user_id', 'post_id', 'target']]

done_test = result_test.merge(user_data, on='user_id', how='left').merge(post_text_df, on='post_id', how='left')
done_test = done_test.drop(columns=['user_id','post_id'])


numeric_cols = done_test.select_dtypes(include=['float64', 'int64']).columns
done_test[numeric_cols] = done_test[numeric_cols].astype('float32')

numeric_cols = done_train.select_dtypes(include=['float64', 'int64']).columns
done_train[numeric_cols] = done_train[numeric_cols].astype('float32')

### Разделение признаков и целевой переменной

На данном этапе формируются итоговые наборы данных для обучения моделей.

Целевая переменная:
- `target` — факт взаимодействия пользователя с постом (лайк / отсутствие лайка).

Признаки:
- `X_train`, `X_test` — подготовленные признаки пользователей, постов и времени;
- `y_train`, `y_test` — целевая переменная для обучения и оценки моделей.

Полученные данные сохраняются в папку `processed` для дальнейшего использования в моделировании.

In [38]:
X_test  = done_test.drop(columns=['target'])
y_test = done_test['target']
X_train  = done_train.drop(columns=['target'])
y_train = done_train['target']

In [39]:
import os

path = "../data/processed"

os.makedirs(path, exist_ok=True)

X_train.to_csv(f"{path}/X_train.csv", index=False)
y_train.to_csv(f"{path}/y_train.csv", index=False)

X_test.to_csv(f"{path}/X_test.csv", index=False)
y_test.to_csv(f"{path}/y_test.csv", index=False)